# Phase 3 — encoder with learned chunk attention

Use the same Phase 1 MHA encoder architecture and masked mean pooling as your Phase 4 full fine-tuning notebook. The encoder loads from `/content/drive/MyDrive/best_enc.pt`. This remains the Phase 3 sentence-role prediction task.

All training, development, test, and vocabulary JSON files come from the official Hugging Face dataset. The tokenizer and encoder checkpoint come from Drive. Set the Colab secret `HF_TOKEN` or enter the token when prompted.

Every sentence is split into consecutive chunks of up to 512 tokens, with every token retained. The frozen encoder creates one vector per chunk. Learned attention combines those vectors into one sentence embedding, and a two-layer BiLSTM reads the document's ordered sentence embeddings to predict the 13 rhetorical roles.

**Speed choices** — fused attention, batches grouped by chunk length, persistent chunk caches, and head-only training. Each completed document is cached, so interrupted encoding resumes with the remaining documents. The encoder is frozen throughout this notebook. The attention scorer, BiLSTM, and classifier are trained.

Progress bars show tokenization, chunk encoding, document training, dev evaluation, and test prediction. Training reports dev Macro-F1 after each epoch and saves an epoch checkpoint on Drive.

The default runs only `attention`. Set `RUN_MODES = ("truncation", "mean", "attention")` for a controlled comparison using the same frozen encoder cache. Dev scores determine which selected run exports `submission.zip`.

[Encoder architecture source](https://github.com/yugan12345/CS781-Capstone-2026/blob/1345ed014ce33cdd2f2aa0354de97a295790a31d/phase-4/cjpe_01_full_finetune_%282%29.ipynb)


## 1. Configuration

Mount Drive and choose the run settings. The default checkpoint is your encoder at the root of MyDrive.

In [1]:
from pathlib import Path
from getpass import getpass
import hashlib
import math
import json
import os
import random
import re
import sys
import zipfile
from collections import Counter
from dataclasses import asdict, dataclass
from contextlib import nullcontext

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import f1_score
from importlib.util import find_spec

if find_spec("huggingface_hub") is None:
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])

from huggingface_hub import HfApi, hf_hub_download
from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")
RR_DIR = DRIVE_ROOT / "Phase_3_RR"
OUTPUT_DIR = DRIVE_ROOT / "Phase_3_RR_Encoder_Attention"

# Set an explicit path if Drive contains multiple matching copies.
CHECKPOINT = "/content/drive/MyDrive/best_enc.pt"
TOKENIZER_JSON = None
TOKENIZER_PY = None

RUN_MODES = ("attention",)
# To compare pooling methods, use ("truncation", "mean", "attention").
RESUME = True
HF_REPO = "Exploration-Lab/CS-781-Capstone"
HF_REVISION = "main"

@dataclass(frozen=True)
class Config:
    vocab_size: int = 40_000
    d_model: int = 768
    n_heads: int = 12
    max_seq_len: int = 512
    rope_theta: float = 10_000.0
    chunk_tokens: int = 512
    encode_batch_size: int = 32
    max_padded_tokens: int = 8192
    bilstm_hidden: int = 256
    epochs: int = 40
    patience: int = 7
    grad_accum: int = 1
    samples_per_doc: int = 1
    head_lr: float = 1e-3
    weight_decay: float = 0.01
    warmup_fraction: float = 0.05
    seed: int = 42

CFG = Config()
MODES = {"truncation", "mean", "attention"}
if not RUN_MODES or len(set(RUN_MODES)) != len(RUN_MODES) or set(RUN_MODES) - MODES:
    raise ValueError("Choose distinct modes from truncation, mean, and attention.")
if not 1 <= CFG.chunk_tokens <= CFG.max_seq_len:
    raise ValueError("chunk_tokens must be between 1 and the Phase 1 context limit.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def amp_context():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16) if USE_AMP else nullcontext()
from tqdm.auto import tqdm


Mounted at /content/drive


## 2. Load assets and data

Download all Phase 3 JSON files from Hugging Face. Load the tokenizer from Drive and preserve the supplied sentence order.

In [2]:
def normalized_name(name):
    return re.sub(r" \(\d+\)(?=\.)", "", name).casefold()

def drive_file(filename, setting, explicit=None, candidates=()):
    if explicit is not None:
        path = Path(explicit)
        path = path if path.is_absolute() else DRIVE_ROOT / path
        if not path.is_file():
            raise FileNotFoundError(f"{setting} does not exist: {path}")
        return path

    for path in candidates:
        if path.is_file():
            return path

    matches = sorted({
        Path(root) / name
        for root, dirs, files in os.walk(DRIVE_ROOT)
        for name in files
        if normalized_name(name) == filename.casefold()
    })
    if len(matches) == 1:
        return matches[0]
    if matches:
        options = "\n".join(str(path) for path in matches)
        raise ValueError(f"Set {setting} to the intended file. Found:\n{options}")
    raise FileNotFoundError(f"Put {filename} on Drive or set {setting}.")

checkpoint_path = drive_file("best_enc.pt", "CHECKPOINT", CHECKPOINT)
tokenizer_json_path = drive_file(
    "bpe_40k.json", "TOKENIZER_JSON", TOKENIZER_JSON,
    (checkpoint_path.parent / "bpe_40k.json",),
)
tokenizer_py_path = drive_file(
    "a1_tokenizer.py", "TOKENIZER_PY", TOKENIZER_PY,
    (checkpoint_path.parent / "a1_tokenizer.py",),
)

try:
    hf_token = userdata.get("HF_TOKEN")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    hf_token = None
hf_token = hf_token or getpass("Hugging Face read token: ").strip()
if not hf_token:
    raise ValueError("Enter a Hugging Face read token with access to the capstone dataset.")

repo_info = HfApi().dataset_info(HF_REPO, revision=HF_REVISION, token=hf_token)
repo_revision = repo_info.sha or HF_REVISION
repo_paths = [item.rfilename for item in (repo_info.siblings or [])]
if not repo_paths:
    repo_paths = HfApi().list_repo_files(HF_REPO, repo_type="dataset", revision=repo_revision, token=hf_token)
phase_files = [
    name for name in repo_paths
    if "phase_3_rr" in name.casefold().split("/")
]
def download_phase3(filename):
    matches = [
        name for name in phase_files
        if normalized_name(Path(name).name) == filename.casefold()
    ]
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} under Phase_3_RR on Hugging Face.")
    if len(matches) > 1:
        raise ValueError(f"Multiple Hugging Face files match {filename}: {matches}")
    return Path(hf_hub_download(
        repo_id=HF_REPO,
        filename=matches[0],
        revision=repo_revision,
        repo_type="dataset",
        token=hf_token,
    ))

domains = ("CL", "IT")
train_paths = {domain: download_phase3(f"{domain}_train.json") for domain in domains}
dev_paths = {domain: download_phase3(f"{domain}_dev.json") for domain in domains}
test_paths = {domain: download_phase3(f"{domain}_test.json") for domain in domains}
label_path = download_phase3("label_vocab.json")
del hf_token

import importlib.util

tokenizer_spec = importlib.util.spec_from_file_location("a1_tokenizer", tokenizer_py_path)
tokenizer_module = importlib.util.module_from_spec(tokenizer_spec)
sys.modules[tokenizer_spec.name] = tokenizer_module
sys.path.insert(0, str(tokenizer_py_path.parent))
tokenizer_spec.loader.exec_module(tokenizer_module)
tokenizer = tokenizer_module.BPETokenizer.load(str(tokenizer_json_path))
if int(tokenizer.vocab_size) != CFG.vocab_size:
    raise ValueError("Use the 40,000-token Phase 1 tokenizer that belongs to this checkpoint.")
PAD_ID = int((getattr(tokenizer, "special_tokens", {}) or {}).get("[PAD]", 0))

def read_json(path):
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)

label_to_id = read_json(label_path)
if not isinstance(label_to_id, dict) or set(label_to_id.values()) != set(range(13)):
    raise ValueError("label_vocab.json must map the 13 official labels to IDs 0 through 12.")
ID_TO_LABEL = {index: name for name, index in label_to_id.items()}
NUM_LABELS = len(ID_TO_LABEL)

@dataclass
class Document:
    doc_id: str
    domain: str
    sentences: list
    labels: list | None

def load_split(paths, labeled):
    docs = []
    for domain, path in paths.items():
        raw = read_json(path)
        if isinstance(raw, list):
            rows = raw
        elif not labeled and isinstance(raw, dict):
            rows = [{"id": doc_id, "text": sentences} for doc_id, sentences in raw.items()]
        else:
            raise ValueError(f"Unexpected document format in {path.name}")
        for row in tqdm(rows, desc=f"Tokenizing {domain} {path.stem}", leave=False):
            doc_id, sentences = str(row["id"]), row["text"]
            if not isinstance(sentences, list) or any(not isinstance(s, str) for s in sentences):
                raise ValueError(f"Expected ordered sentence strings for {doc_id}.")
            labels = [label_to_id[label] for label in row["labels"]] if labeled else None
            if labeled and (not labels or len(labels) != len(sentences)):
                raise ValueError(f"Sentence and label counts differ for {doc_id}.")
            tokens = [list(tokenizer.encode(sentence)) or [PAD_ID] for sentence in sentences]
            docs.append(Document(doc_id, domain, tokens, labels))
    if not docs or len({doc.doc_id for doc in docs}) != len(docs):
        raise ValueError("A split is empty or contains duplicate document IDs.")
    return docs

train_docs = load_split(train_paths, labeled=True)
dev_docs = load_split(dev_paths, labeled=True)
test_docs = load_split(test_paths, labeled=False)
counts = Counter(label for doc in train_docs for label in doc.labels)
weights = torch.tensor([max(1, counts[i]) ** -0.5 for i in range(NUM_LABELS)])
class_weights = (weights / weights.mean()).clamp(max=3.0).to(DEVICE)

for name, docs in (("Train", train_docs), ("Dev", dev_docs), ("Test", test_docs)):
    lengths = [len(tokens) for doc in docs for tokens in doc.sentences]
    long_count = sum(length > CFG.chunk_tokens for length in lengths)
    print(f"{name}: {len(docs)} documents, {len(lengths)} sentences, "
          f"{long_count} sentences over {CFG.chunk_tokens} tokens")

def sha256_file(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

input_files = [*train_paths.values(), *dev_paths.values(), label_path,
               tokenizer_json_path, tokenizer_py_path]
checkpoint_stat = checkpoint_path.stat()
run_config = {
    "version": 2,
    "config": asdict(CFG),
    "checkpoint": [str(checkpoint_path), checkpoint_stat.st_size, checkpoint_stat.st_mtime_ns],
    "data_hashes": [sha256_file(path) for path in input_files],
}
RUN_ID = hashlib.sha256(json.dumps(run_config, sort_keys=True).encode()).hexdigest()[:12]
RUN_DIR = OUTPUT_DIR / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"Outputs: {RUN_DIR}")

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

... (4 outputs left out)

Tokenizing IT IT_dev:   0%|          | 0/5 [00:00<?, ?it/s]

Tokenizing CL CL_test:   0%|          | 0/5 [00:00<?, ?it/s]

Tokenizing IT IT_test:   0%|          | 0/5 [00:00<?, ?it/s]

Train: 80 documents, 16845 sentences, 5 sentences over 512 tokens
Dev: 10 documents, 2142 sentences, 0 sentences over 512 tokens
Test: 10 documents, 2197 sentences, 1 sentences over 512 tokens
Outputs: /content/drive/MyDrive/Phase_3_RR_Encoder_Attention/314b079319e8


## 3. Load the Phase 1 encoder

The architecture matches your Phase 4 MHA encoder. Fused scaled dot-product attention computes the same bidirectional attention with a padding mask. Every required backbone weight must match the checkpoint.

In [3]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * rms).to(x.dtype) * self.weight

def precompute_rope(dim, max_len, theta=10000.0, device=None):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2, device=device).float() / dim))
    t = torch.arange(max_len, device=device).float()
    ang = torch.outer(t, freqs)
    return ang.cos(), ang.sin()

def apply_rope(x, cos, sin):
    T = x.size(2)
    c = cos[:T].unsqueeze(0).unsqueeze(0)
    s = sin[:T].unsqueeze(0).unsqueeze(0)
    hd = x.size(-1) // 2
    x1, x2 = x[..., :hd], x[..., hd:]
    rot = torch.cat([-x2, x1], dim=-1)
    c_e = torch.cat([c.expand(-1, -1, -1, hd), c.expand(-1, -1, -1, hd)], -1)
    s_e = torch.cat([s.expand(-1, -1, -1, hd), s.expand(-1, -1, -1, hd)], -1)
    return x * c_e + rot * s_e

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.nh, self.hd, self.dm = cfg.n_heads, cfg.head_dim, cfg.d_model
        self.q = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.v = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.o = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.attn_drop = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self.scale = 1.0 / math.sqrt(cfg.head_dim)
    def forward(self, x, cos, sin, padding_mask=None):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        k = self.k(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        v = self.v(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        mask = None if padding_mask is None else ~padding_mask[:, None, None, :]
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=0.0)
        out = out.transpose(1, 2).contiguous().view(B, T, self.dm)
        return self.resid_drop(self.o(out))

class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.up   = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.drop = nn.Dropout(cfg.dropout)
    def forward(self, x):
        return self.drop(self.down(F.silu(self.gate(x)) * self.up(x)))

class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model); self.attn = MultiHeadSelfAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model); self.ffn  = SwiGLU(cfg)
    def forward(self, x, cos, sin, padding_mask=None):
        x = x + self.attn(self.norm1(x), cos, sin, padding_mask)
        return x + self.ffn(self.norm2(x))

class BackboneCfg:
    pass

class LegalEncoder(nn.Module):
    """Phase 1 encoder without the MLM head."""
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.layers = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model)
        cos, sin = precompute_rope(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init)   # only matters for tensors absent from the ckpt

    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, ids, padding_mask=None):
        T = ids.size(1)
        if T > self.rope_cos.size(0):
            raise ValueError(
                f"sequence length {T} exceeds the RoPE table ({self.rope_cos.size(0)}). "
                f"Phase 1 precomputed RoPE at max_seq_len only; longer context needs "
                f"the buffers rebuilt and position interpolation.")
        x = self.drop(self.tok_emb(ids))
        for layer in self.layers:
            x = layer(x, self.rope_cos, self.rope_sin, padding_mask)
        return self.norm(x)
def load_encoder():
    payload = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    raw = payload.get("model", payload)
    state = {name.removeprefix("module."): tensor for name, tensor in raw.items()}
    width = state["tok_emb.weight"].shape[1]
    if state["layers.0.attn.k.weight"].shape != (width, width):
        raise ValueError("Select the Phase 1 MHA encoder checkpoint best_enc.pt.")
    if "lm_head.weight" in state and "mlm_dense.weight" not in state:
        raise ValueError("Select the Phase 1 MLM encoder checkpoint best_enc.pt.")
    config = BackboneCfg()
    config.d_model = width
    config.vocab_size = state["tok_emb.weight"].shape[0]
    config.n_layers = len({int(name.split(".")[1]) for name in state if name.startswith("layers.")})
    config.d_ff = state["layers.0.ffn.gate.weight"].shape[0]
    config.n_heads = CFG.n_heads
    config.head_dim = width // config.n_heads
    config.max_seq_len = CFG.max_seq_len
    config.rope_theta = CFG.rope_theta
    config.dropout = 0.0
    if width != CFG.d_model or width % config.n_heads or config.head_dim % 2:
        raise ValueError("Check d_model and n_heads against the Phase 1 encoder.")
    if config.vocab_size < int(tokenizer.vocab_size):
        raise ValueError("The tokenizer vocabulary exceeds the encoder embedding table.")
    encoder = LegalEncoder(config)
    required = encoder.state_dict()
    encoder.load_state_dict({name: state[name] for name in required}, strict=True)
    encoder.requires_grad_(False)
    return encoder.to(DEVICE).eval()

def atomic_save(payload, path):
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary)
    temporary.replace(path)


## 4. Cache every chunk

This is the expensive first pass. Cached documents are reused across reruns and pooling modes. Changing the checkpoint, tokenizer, or chunk size selects a separate cache.

In [4]:
cache_identity = {
    "version": 1,
    "checkpoint": run_config["checkpoint"],
    "tokenizer": [sha256_file(tokenizer_json_path), sha256_file(tokenizer_py_path)],
    "chunk_tokens": CFG.chunk_tokens,
    "heads": CFG.n_heads,
    "rope_theta": CFG.rope_theta,
    "pooling": "masked_mean_no_added_special_tokens",
}
cache_id = hashlib.sha256(json.dumps(cache_identity, sort_keys=True).encode()).hexdigest()[:16]
CACHE_DIR = OUTPUT_DIR / "encoder_cache" / cache_id
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def cache_path(doc):
    digest = hashlib.sha256(json.dumps(doc.sentences, separators=(",", ":")).encode()).hexdigest()
    return CACHE_DIR / f"{digest}.pt"

def document_chunks(doc):
    chunks, counts = [], []
    for tokens in doc.sentences:
        parts = [tokens[start:start + CFG.chunk_tokens]
                 for start in range(0, len(tokens), CFG.chunk_tokens)]
        counts.append(len(parts))
        chunks.extend(parts)
    return chunks, counts

@torch.inference_mode()
def encode_document(encoder, doc, progress):
    chunks, counts = document_chunks(doc)
    values = torch.empty((len(chunks), CFG.d_model), dtype=torch.float16)
    order = sorted(range(len(chunks)), key=lambda index: len(chunks[index]))
    start = 0
    while start < len(order):
        end = min(start + CFG.encode_batch_size, len(order))
        while (end - start) * len(chunks[order[end - 1]]) > CFG.max_padded_tokens and end > start + 1:
            end -= 1
        indices = order[start:end]
        lengths = torch.tensor([len(chunks[i]) for i in indices], device=DEVICE)
        ids = pad_sequence(
            [torch.tensor(chunks[i], dtype=torch.long) for i in indices],
            batch_first=True, padding_value=PAD_ID,
        ).to(DEVICE)
        valid = torch.arange(ids.size(1), device=DEVICE)[None, :] < lengths[:, None]
        with amp_context():
            hidden = encoder(ids, padding_mask=~valid)
        vectors = (hidden.float() * valid.unsqueeze(-1)).sum(1) / lengths.unsqueeze(1)
        values[indices] = vectors.cpu().half()
        progress.update(len(indices))
        start = end
    return {"values": values, "counts": counts}

all_docs = train_docs + dev_docs + test_docs
pending = [doc for doc in all_docs if not cache_path(doc).is_file()]
if pending:
    encoder = load_encoder()
    total_chunks = sum(sum((len(tokens) + CFG.chunk_tokens - 1) // CFG.chunk_tokens
                           for tokens in doc.sentences) for doc in pending)
    with tqdm(total=total_chunks, desc="Encoding and caching chunks", unit="chunk") as progress:
        for doc in pending:
            features = encode_document(encoder, doc, progress)
            atomic_save(features, cache_path(doc))
    del encoder

for doc in tqdm(all_docs, desc="Loading cached embeddings", unit="doc"):
    doc.features = torch.load(cache_path(doc), map_location="cpu", weights_only=True)
    if len(doc.features["counts"]) != len(doc.sentences):
        raise ValueError(f"Cached sentence count differs for {doc.doc_id}.")
print(f"Chunk embeddings ready for {len(all_docs)} documents.")


Encoding and caching chunks:   0%|          | 0/21191 [00:00<?, ?chunk/s]

Loading cached embeddings:   0%|          | 0/100 [00:00<?, ?doc/s]

Chunk embeddings ready for 100 documents.


## 5. Learned chunk pooling

The scorer starts with equal weights and learns from sentence labels. Single-chunk sentences receive weight 1. The BiLSTM receives the complete sequence of sentence embeddings.

In [5]:
class ChunkAttention(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.projection = nn.Linear(width + 2, 128)
        self.score = nn.Linear(128, 1, bias=False)
        nn.init.zeros_(self.score.weight)

    def forward(self, embeddings, counts):
        values = pad_sequence(embeddings.split(counts), batch_first=True)
        sizes = torch.tensor(counts, device=values.device).unsqueeze(1)
        steps = torch.arange(values.size(1), device=values.device).unsqueeze(0)
        valid = steps < sizes
        relative = (steps / (sizes - 1).clamp_min(1)).to(values.dtype)
        positions = torch.stack((relative, 1 - relative), dim=-1)
        scores = self.score(torch.tanh(self.projection(torch.cat((values, positions), -1))))
        weights = scores.squeeze(-1).float().masked_fill(~valid, float("-inf")).softmax(1)
        return (values * weights.unsqueeze(-1)).sum(1), weights

class SentenceRoleModel(nn.Module):
    def __init__(self, mode):
        super().__init__()
        self.mode = mode
        self.norm = nn.LayerNorm(CFG.d_model)
        self.bilstm = nn.LSTM(
            CFG.d_model, CFG.bilstm_hidden, num_layers=2,
            batch_first=True, bidirectional=True, dropout=0.3,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.4), nn.Linear(2 * CFG.bilstm_hidden, CFG.bilstm_hidden),
            nn.GELU(), nn.Dropout(0.4), nn.Linear(CFG.bilstm_hidden, NUM_LABELS),
        )
        self.attention = ChunkAttention(CFG.d_model) if mode == "attention" else None

    def forward(self, features):
        values = features["values"].to(DEVICE, dtype=torch.float32)
        counts = features["counts"]
        weights = None
        if self.mode == "attention":
            pooled, weights = self.attention(values, counts)
        elif self.mode == "mean":
            pooled = torch.stack([group.mean(0) for group in values.split(counts)])
        else:
            pooled = torch.stack([group[0] for group in values.split(counts)])
        contextual, _ = self.bilstm(self.norm(pooled).unsqueeze(0))
        return self.classifier(contextual.squeeze(0)), weights, counts

def build_model(mode):
    seed_all(CFG.seed)
    return SentenceRoleModel(mode).to(DEVICE)

def trainable_state(model):
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}

def load_trainable_state(model, state):
    model.load_state_dict(state, strict=True)


## 6. Train with live progress

Use all training documents once per epoch, select the best dev Macro-F1, and stop after seven epochs without improvement. Rerunning resumes at the last saved epoch.

In [6]:
criterion = nn.CrossEntropyLoss(weight=class_weights)

@torch.inference_mode()
def evaluate(model, docs):
    model.eval()
    gold, predicted, domains, long_flags = [], [], [], []
    for doc in tqdm(docs, desc="Evaluating", unit="doc", leave=False):
        with nullcontext():
            logits, _, _ = model(doc.features)
        gold.extend(doc.labels)
        predicted.extend(logits.argmax(dim=-1).tolist())
        domains.extend([doc.domain] * len(doc.labels))
        long_flags.extend(len(tokens) > CFG.chunk_tokens for tokens in doc.sentences)

    gold, predicted = np.asarray(gold), np.asarray(predicted)
    domains, long_flags = np.asarray(domains), np.asarray(long_flags)
    all_ids = list(range(NUM_LABELS))

    def macro(mask):
        if not mask.any():
            return None
        return float(f1_score(gold[mask], predicted[mask], labels=all_ids,
                              average="macro", zero_division=0))

    long_total = int(long_flags.sum())
    long_correct = int((gold[long_flags] == predicted[long_flags]).sum())
    long_gold_labels = np.unique(gold[long_flags])
    long_f1 = (
        float(f1_score(gold[long_flags], predicted[long_flags], labels=long_gold_labels,
                       average="macro", zero_division=0))
        if long_total else None
    )
    return {
        "macro_f1": macro(np.ones(len(gold), dtype=bool)),
        "cl_macro_f1": macro(domains == "CL"),
        "it_macro_f1": macro(domains == "IT"),
        "long_correct": long_correct,
        "long_total": long_total,
        "long_macro_f1_supported_labels": long_f1,
        "long_supported_labels": [ID_TO_LABEL[int(label)] for label in long_gold_labels],
    }

def make_optimizer(model):
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=CFG.head_lr, weight_decay=CFG.weight_decay,
    )
    per_epoch = len(train_docs) * CFG.samples_per_doc
    total_steps = CFG.epochs * ((per_epoch + CFG.grad_accum - 1) // CFG.grad_accum)
    warmup = max(1, int(total_steps * CFG.warmup_fraction))

    def multiplier(step):
        if step < warmup:
            return step / warmup
        progress = min(1.0, (step - warmup) / max(1, total_steps - warmup))
        return max(0.01, 0.5 * (1 + math.cos(math.pi * progress)))

    return optimizer, torch.optim.lr_scheduler.LambdaLR(optimizer, multiplier)

def train_epoch(model, optimizer, scheduler, scaler, epoch):
    model.train()
    seed_all(CFG.seed + epoch)
    order = list(range(len(train_docs))) * CFG.samples_per_doc
    random.Random(CFG.seed + epoch).shuffle(order)
    trainable = [param for param in model.parameters() if param.requires_grad]
    total_loss = 0.0
    progress = tqdm(range(0, len(order), CFG.grad_accum), desc=f"{model.mode} epoch {epoch}", unit="step")
    for start in progress:
        group = order[start:start + CFG.grad_accum]
        optimizer.zero_grad(set_to_none=True)
        for index in group:
            doc = train_docs[index]
            targets = torch.as_tensor(doc.labels, dtype=torch.long, device=DEVICE)
            with nullcontext():
                logits, _, _ = model(doc.features)
                loss = criterion(logits, targets)
            scaler.scale(loss / len(group)).backward()
            total_loss += loss.detach().item()
        progress.set_postfix(loss=f"{total_loss / (start + len(group)):.4f}")
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(trainable, 1.0)
        old_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        if scaler.get_scale() >= old_scale:
            scheduler.step()
    return total_loss / len(order)

def atomic_save(payload, path):
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary)
    temporary.replace(path)

def fit_mode(mode):
    model = build_model(mode)
    optimizer, scheduler = make_optimizer(model)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=False)
    state_path = RUN_DIR / f"{mode}.pt"
    history = []
    best = None
    best_weights = None
    bad_epochs = 0
    if RESUME and state_path.is_file():
        saved = torch.load(state_path, map_location="cpu", weights_only=True)
        load_trainable_state(model, saved["latest_weights"])
        optimizer.load_state_dict(saved["optimizer"])
        scheduler.load_state_dict(saved["scheduler"])
        scaler.load_state_dict(saved["scaler"])
        history, best = saved["history"], saved["best"]
        best_weights, bad_epochs = saved["best_weights"], saved["bad_epochs"]
        del saved
        print(f"{mode}: restored epoch {len(history)}")

    for epoch in range(len(history) + 1, CFG.epochs + 1):
        if bad_epochs >= CFG.patience:
            break
        train_loss = train_epoch(model, optimizer, scheduler, scaler, epoch)
        metrics = {"epoch": epoch, "train_loss": train_loss, **evaluate(model, dev_docs)}
        history.append(metrics)
        print(f"{mode} | epoch {epoch:02d} | loss {train_loss:.4f} | "
              f"dev Macro-F1 {metrics['macro_f1']:.4f} | "
              f"long sentences {metrics['long_correct']}/{metrics['long_total']} correct")
        latest_weights = trainable_state(model)
        if best is None or metrics["macro_f1"] > best["macro_f1"]:
            best, best_weights, bad_epochs = metrics, latest_weights, 0
        else:
            bad_epochs += 1
        atomic_save({
            "latest_weights": latest_weights,
            "best_weights": best_weights,
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "history": history,
            "best": best,
            "bad_epochs": bad_epochs,
        }, state_path)
    return {"mode": mode, **best, "history": history}

results = [fit_mode(mode) for mode in RUN_MODES]


attention epoch 1:   0%|          | 0/80 [00:00<?, ?step/s]

Evaluating:   0%|          | 0/10 [00:00<?, ?doc/s]

attention | epoch 01 | loss 2.3793 | dev Macro-F1 0.0919 | long sentences 0/0 correct


attention epoch 2:   0%|          | 0/80 [00:00<?, ?step/s]

Evaluating:   0%|          | 0/10 [00:00<?, ?doc/s]

attention | epoch 02 | loss 1.9590 | dev Macro-F1 0.1744 | long sentences 0/0 correct


... (80 outputs left out)

attention | epoch 29 | loss 0.0471 | dev Macro-F1 0.4163 | long sentences 0/0 correct


attention epoch 30:   0%|          | 0/80 [00:00<?, ?step/s]

Evaluating:   0%|          | 0/10 [00:00<?, ?doc/s]

attention | epoch 30 | loss 0.0370 | dev Macro-F1 0.4130 | long sentences 0/0 correct


## 7. Evaluate and export

Export one label for every test sentence. The comparison includes CL, IT, and the long-sentence subset.

In [7]:
def format_metric(value):
    return "—" if value is None else f"{value:.4f}"

for result in results:
    print(
        f"{result['mode']:10} epoch {result['epoch']:2d} | "
        f"dev {result['macro_f1']:.4f} | "
        f"CL {format_metric(result['cl_macro_f1'])} | "
        f"IT {format_metric(result['it_macro_f1'])} | "
        f"long {result['long_correct']}/{result['long_total']} correct | "
        f"long supported-label F1 {format_metric(result['long_macro_f1_supported_labels'])}"
    )

winner = max(results, key=lambda result: result["macro_f1"])
summary = {
    "hugging_face_revision": repo_revision,
    "inputs": run_config,
    "selected_mode": winner["mode"],
    "results": results,
}
(RUN_DIR / "comparison.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

def restore_best(mode):
    model = build_model(mode)
    saved = torch.load(RUN_DIR / f"{mode}.pt", map_location="cpu", weights_only=True)
    load_trainable_state(model, saved["best_weights"])
    return model.eval()

@torch.inference_mode()
def predict_test(model):
    output = {}
    for doc in tqdm(test_docs, desc="Predicting test", unit="doc"):
        if doc.sentences:
            with nullcontext():
                logits, _, _ = model(doc.features)
            output[doc.doc_id] = [ID_TO_LABEL[index] for index in logits.argmax(dim=-1).tolist()]
        else:
            output[doc.doc_id] = []
        assert len(output[doc.doc_id]) == len(doc.sentences)
    return output

model = restore_best(winner["mode"])
predictions = predict_test(model)
del model
prediction_path = RUN_DIR / "prediction.json"
prediction_path.write_text(json.dumps(predictions, ensure_ascii=False, indent=2), encoding="utf-8")
with zipfile.ZipFile(RUN_DIR / "submission.zip", "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(prediction_path, arcname="prediction.json")
print(f"Selected by dev Macro-F1: {winner['mode']}")
print(f"Submission: {RUN_DIR / 'submission.zip'}")


attention  epoch 23 | dev 0.4401 | CL 0.4297 | IT 0.4097 | long 0/0 correct | long supported-label F1 —


Predicting test:   0%|          | 0/10 [00:00<?, ?doc/s]

Selected by dev Macro-F1: attention
Submission: /content/drive/MyDrive/Phase_3_RR_Encoder_Attention/314b079319e8/submission.zip


## 8. Inspect chunk weights

Save each long dev sentence's chunk ranges, attention weights, gold role, and predicted role.

In [8]:
@torch.inference_mode()
def inspect_attention(model):
    examples = []
    for doc in tqdm(dev_docs, desc="Inspecting chunk weights", unit="doc"):
        if not any(len(tokens) > CFG.chunk_tokens for tokens in doc.sentences):
            continue
        with nullcontext():
            logits, weights, counts = model(doc.features)
        predictions = logits.argmax(dim=-1).tolist()
        weights = weights.cpu().tolist()
        for i, (tokens, count) in enumerate(zip(doc.sentences, counts)):
            if count == 1:
                continue
            chunks = [
                {
                    "start_token": j * CFG.chunk_tokens + 1,
                    "end_token": min((j + 1) * CFG.chunk_tokens, len(tokens)),
                    "weight": weights[i][j],
                }
                for j in range(count)
            ]
            examples.append({
                "document_id": doc.doc_id,
                "domain": doc.domain,
                "sentence_index": i + 1,
                "tokens": len(tokens),
                "gold_label": ID_TO_LABEL[doc.labels[i]],
                "predicted_label": ID_TO_LABEL[predictions[i]],
                "highest_weight_chunk": max(chunks, key=lambda chunk: chunk["weight"]),
                "chunks": chunks,
            })
    return examples

if "attention" in RUN_MODES:
    attention_model = restore_best("attention")
    examples = inspect_attention(attention_model)
    del attention_model
    path = RUN_DIR / "dev_chunk_attention.json"
    path.write_text(json.dumps(examples, indent=2), encoding="utf-8")
    print(f"Saved chunk weights for {len(examples)} long dev sentences: {path}")


Inspecting chunk weights:   0%|          | 0/10 [00:00<?, ?doc/s]

Saved chunk weights for 0 long dev sentences: /content/drive/MyDrive/Phase_3_RR_Encoder_Attention/314b079319e8/dev_chunk_attention.json
